<a href="https://colab.research.google.com/github/R3beAM/Proyecto-Final-Integracion/blob/second/Evaluacion_2_1_Rebeca_Alvarez.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#**Descripción del dataset original**

In [ ]:
import pandas as pd
import numpy as np

import nltk
from nltk.corpus import stopwords

import re
import string

In [ ]:
# Ruta al archivo (ajusta el nombre del archivo)
file_path = "/content/Amazon_Consumer_Review.csv"

# =========================
# 1. Cargar dataset
# =========================

df = pd.read_csv("Amazon_Consumer_Review.csv")

print("DESCRIPCIÓN GENERAL DEL DATASET")
print("-" * 50)
print("Filas y columnas:", df.shape)
print("Cantidad de filas:", df.shape[0])
print("Cantidad de columnas:", df.shape[1])

print("\nColumnas del dataset:")
print(df.columns.tolist())

print("\nPrimeras filas:")
display(df.head())

print("\nInformación general:")
df.info()

##1. Resumen de tipos de variables

In [ ]:
# =========================
# 2. Tipos de variables
# =========================

tipos_variables = pd.DataFrame({
    "Variable": df.columns,
    "Tipo de dato": df.dtypes.astype(str),
    "Valores no nulos": df.notnull().sum().values,
    "Valores nulos": df.isnull().sum().values,
    "Porcentaje nulos": ((df.isnull().sum() / len(df)) * 100).round(2).values,
    "Valores únicos": df.nunique(dropna=False).values
})

display(tipos_variables)

##Análisis de variables altamente correlacionadas

(Los problemas de cardinalidad alta, desbalance de clases y escalas inconsistentes se tomaron del avance anterior, y se documentaron en el documento de word)



In [ ]:
import pandas as pd

df = pd.read_csv("Amazon_Consumer_Review.csv")

# =========================
# 3. Crear variables numéricas auxiliares
# =========================

df["reviews.numHelpful"] = pd.to_numeric(df["reviews.numHelpful"], errors="coerce")

df["review_length_chars"] = df["reviews.text"].astype(str).apply(len)
df["review_length_words"] = df["reviews.text"].astype(str).apply(lambda x: len(x.split()))

# Variables numéricas disponibles
numeric_df = df.select_dtypes(include=["int64", "float64"])

print("Variables numéricas detectadas:")
print(numeric_df.columns.tolist())

# =========================
# 4. Matriz de correlación
# =========================

correlation_matrix = numeric_df.corr()

display(correlation_matrix.round(3))

# =========================
# 5. Identificar correlaciones altas
# =========================

correlation_pairs = correlation_matrix.unstack().reset_index()
correlation_pairs.columns = ["Variable 1", "Variable 2", "Correlación"]

# Eliminar correlación consigo misma
correlation_pairs = correlation_pairs[
    correlation_pairs["Variable 1"] != correlation_pairs["Variable 2"]
]

# Evitar duplicados
correlation_pairs["Par ordenado"] = correlation_pairs.apply(
    lambda row: tuple(sorted([row["Variable 1"], row["Variable 2"]])),
    axis=1
)

correlation_pairs = correlation_pairs.drop_duplicates("Par ordenado")
correlation_pairs = correlation_pairs.drop(columns=["Par ordenado"])

# Filtrar correlaciones fuertes
high_correlations = correlation_pairs[
    correlation_pairs["Correlación"].abs() >= 0.70
].sort_values(by="Correlación", ascending=False)

print("Variables altamente correlacionadas:")
display(high_correlations.round(3))

In [ ]:
#Vamos a modificar la columna reviews.username a que sea string, para que pueda ser parte de las correlaciones
# Reemplazar nulos y convertir a texto
df["reviews.username"] = df["reviews.username"].fillna("unknown").astype("string")

# Verificar resultado
print(df["reviews.username"].dtype)
df[["reviews.username"]].head()

In [ ]:
# Convertir reviews.username a tipo texto/string
df["reviews.username"] = df["reviews.username"].astype("string")

# Verificar el tipo de dato
print(df["reviews.username"].dtype)

# Ver primeras filas
df[["reviews.username"]].head()

In [ ]:
# =========================
# 6. Asociación entre variables de texto (Cramér's V + coincidencia + tablas cruzadas)
# =========================

from scipy.stats import chi2_contingency
import numpy as np
import itertools

text_cols = ["brand", "manufacturer", "reviews.username"]

# Asegurar columnas como texto y sin nulos
for col in text_cols:
    df[col] = df[col].fillna("missing").astype("string")


def cramers_v(x, y):
    """Calcula Cramér's V con corrección de sesgo de Bergsma/Wicher."""
    contingency = pd.crosstab(x, y)

    if contingency.empty:
        return np.nan

    chi2, _, _, _ = chi2_contingency(contingency)
    n = contingency.to_numpy().sum()

    if n == 0:
        return np.nan

    phi2 = chi2 / n
    r, k = contingency.shape

    # Corrección por sesgo
    phi2corr = max(0, phi2 - ((k - 1) * (r - 1)) / (n - 1))
    rcorr = r - ((r - 1) ** 2) / (n - 1)
    kcorr = k - ((k - 1) ** 2) / (n - 1)

    denom = min((kcorr - 1), (rcorr - 1))
    if denom <= 0:
        return np.nan

    return np.sqrt(phi2corr / denom)


results = []
for a, b in itertools.combinations(text_cols, 2):
    pair_df = df[[a, b]].dropna()

    # Cramér's V
    c_v = cramers_v(pair_df[a], pair_df[b])

    # % de coincidencia exacta (útil especialmente para brand vs manufacturer)
    match_pct = (pair_df[a].str.lower().str.strip() == pair_df[b].str.lower().str.strip()).mean() * 100

    # Tabla cruzada
    ctab = pd.crosstab(pair_df[a], pair_df[b])

    results.append(
        {
            "Variable 1": a,
            "Variable 2": b,
            "Cramér's V": c_v,
            "% coincidencia": match_pct,
            "Tabla cruzada": ctab,
        }
    )

# Resumen numérico
summary_df = pd.DataFrame(
    [
        {
            "Variable 1": r["Variable 1"],
            "Variable 2": r["Variable 2"],
            "Cramér's V": r["Cramér's V"],
            "% coincidencia": r["% coincidencia"],
        }
        for r in results
    ]
)

print("Resumen de asociación entre variables categóricas:")
display(summary_df.sort_values("Cramér's V", ascending=False).round(4))

# Mostrar tablas cruzadas para revisar posibles réplicas
for r in results:
    print(f"\nTabla cruzada: {r['Variable 1']} vs {r['Variable 2']}")
    display(r["Tabla cruzada"])



**Interpretación de los resultados**

Estos resultados analizan la asociación entre variables categóricas usando Cramér’s V y porcentaje de coincidencia. En general, muestran que existe una relación fuerte entre brand y manufacturer, mientras que las asociaciones con reviews.username deben interpretarse con mucha cautela por su alta cardinalidad.

1. Asociación entre brand y manufacturer

| Variable 1 | Variable 2     | Cramér’s V | % coincidencia |
| ---------- | -------------- | ---------: | -------------: |
| `brand`    | `manufacturer` |     0.7071 |       99.9188% |


El valor de Cramér’s V = 0.7071 indica una asociación alta entre brand y manufacturer.

Además, el porcentaje de coincidencia es de 99.9188%, lo cual significa que ambas variables contienen prácticamente la misma información en casi todos los registros.

La tabla cruzada confirma esto:

| brand        | Amazon | Amazon Digital Services | Amazon.com | AmazonBasics |
| ------------ | -----: | ----------------------: | ---------: | -----------: |
| Amazon       |  16130 |                      18 |          5 |            0 |
| AmazonBasics |      0 |                       0 |          0 |           10 |
| Amazonbasics |      0 |                       0 |          0 |        12169 |


La mayoría de registros con brand = Amazon tienen manufacturer = Amazon. De igual forma, brand = Amazonbasics se asocia casi completamente con manufacturer = AmazonBasics.

Conclusión

brand y manufacturer son variables altamente redundantes.

Para el modelo, no es necesario usar ambas. Se recomienda conservar solo una para evitar duplicidad de información.

Para mi modelo, conservaré brand, porque es más clara desde el punto de vista de negocio y suele representar mejor la identidad comercial del producto.

También conviene normalizar valores como:

AmazonBasics
Amazonbasics

porque representan probablemente la misma marca, pero están escritos con diferente capitalización.

2. Asociación entre brand y reviews.username

| Variable 1 | Variable 2         | Cramér’s V | % coincidencia |
| ---------- | ------------------ | ---------: | -------------: |
| `brand`    | `reviews.username` |     0.6524 |        0.0035% |


A primera vista, Cramér’s V = 0.6524 parece indicar una asociación moderada-alta. Sin embargo, el porcentaje de coincidencia es prácticamente cero: 0.0035%.

Esto ocurre porque reviews.username tiene una cardinalidad extremadamente alta: la tabla muestra 16,269 columnas, una por cada usuario.

Cuando una variable categórica tiene muchísimos valores únicos, Cramér’s V puede dar una impresión inflada de asociación, especialmente si muchos usuarios aparecen pocas veces o solo asociados a una marca por casualidad.

En otras palabras, la asociación no necesariamente significa que el usuario explique la marca o que sea útil para el modelo. Puede ser un efecto estadístico causado por la alta cardinalidad.

Conclusión

reviews.username no debe considerarse una variable útil para el modelo.

Aunque Cramér’s V parece alto, la variable tiene demasiadas categorías, baja generalización y puede introducir ruido.

En mi modelo voy a excluir reviews.username del modelo predictivo.

Razones:

alta cardinalidad;
poco valor predictivo generalizable;
riesgo de sobreajuste;
posible problema de privacidad;
no aporta lógica de negocio directa para predecir caída en desempeño.

3. Asociación entre manufacturer y reviews.username

| Variable 1     | Variable 2         | Cramér’s V | % coincidencia |
| -------------- | ------------------ | ---------: | -------------: |
| `manufacturer` | `reviews.username` |     0.6129 |        0.0035% |


Interpretación

El caso es similar al anterior. El valor de Cramér’s V = 0.6129 puede parecer alto, pero está influenciado por la alta cardinalidad de reviews.username.

La tabla cruzada muestra muchísimos usuarios, y casi todos se concentran bajo manufacturer = Amazon, porque el dataset está dominado por productos de Amazon.

Esto no significa que reviews.username sea una buena variable predictiva. Significa más bien que muchos usuarios aparecen en un dataset donde casi todos los productos pertenecen a Amazon o AmazonBasics.

Conclusión

La asociación entre manufacturer y reviews.username no debe usarse como argumento para conservar reviews.username.

La relación es poco útil desde el punto de vista predictivo y puede generar sobreajuste.

4. Hallazgos principales

| Hallazgo                                              | Interpretación                       | Decisión recomendada               |
| ----------------------------------------------------- | ------------------------------------ | ---------------------------------- |
| `brand` y `manufacturer` están altamente asociados    | Contienen información casi duplicada | Conservar solo una                 |
| `brand` y `manufacturer` tienen 99.9% de coincidencia | Son variables redundantes            | Preferir `brand`                   |
| `reviews.username` tiene cardinalidad extrema         | Más de 16,000 usuarios únicos        | Excluir del modelo                 |
| Cramér’s V con `reviews.username` parece alto         | Puede estar inflado por cardinalidad | No interpretarlo como señal fuerte |
| `AmazonBasics` y `Amazonbasics` aparecen separados    | Problema de consistencia textual     | Estandarizar capitalización        |


5. Implicación para mi modelo

Si incluye brand, manufacturer y reviews.username al mismo tiempo, puede generar:

redundancia;
demasiadas columnas por One-Hot Encoding;
sobreajuste;
baja interpretabilidad;
aumento innecesario de dimensionalidad.

La Regresión Logística asigna un coeficiente a cada variable. Si se incluyen variables redundantes, como brand y manufacturer, los coeficientes pueden volverse menos estables. Si se incluye reviews.username, el modelo puede aprender patrones específicos de usuarios que no se repiten en nuevos datos.

6. Decisión final

| Variable           | Decisión  | Justificación                                             |
| ------------------ | --------- | --------------------------------------------------------- |
| `brand`            | Conservar | Clara, interpretable y útil para segmentación.            |
| `manufacturer`     | Eliminar  | Redundante con `brand`.                                   |
| `reviews.username` | Eliminar  | Alta cardinalidad, riesgo de sobreajuste y baja utilidad. |
.


#Graficos de visualización

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from scipy.stats import chi2_contingency
import numpy as np
import itertools

# --- Code to define summary_df ---
text_cols = ["brand", "manufacturer", "reviews.username"]

# Asegurar columnas como texto y sin nulos
for col in text_cols:
    df[col] = df[col].fillna("missing").astype("string")

def cramers_v(x, y):
    """Calcula Cramér's V con corrección de sesgo de Bergsma/Wicher."""
    contingency = pd.crosstab(x, y)

    if contingency.empty:
        return np.nan

    chi2, _, _, _ = chi2_contingency(contingency)
    n = contingency.to_numpy().sum()

    if n == 0:
        return np.nan

    phi2 = chi2 / n
    r, k = contingency.shape

    # Corrección por sesgo
    phi2corr = max(0, phi2 - ((k - 1) * (r - 1)) / (n - 1))
    rcorr = r - ((r - 1) ** 2) / (n - 1)
    kcorr = k - ((k - 1) ** 2) / (n - 1)

    denom = min((kcorr - 1), (rcorr - 1))
    if denom <= 0:
        return np.nan

    return np.sqrt(phi2corr / denom)


results = []
for a, b in itertools.combinations(text_cols, 2):
    pair_df = df[[a, b]].dropna()

    # Cramér's V
    c_v = cramers_v(pair_df[a], pair_df[b])

    # % de coincidencia exacta (útil especialmente para brand vs manufacturer)
    match_pct = (pair_df[a].str.lower().str.strip() == pair_df[b].str.lower().str.strip()).mean() * 100

    # Tabla cruzada (not strictly needed for summary_df, but part of original logic)
    ctab = pd.crosstab(pair_df[a], pair_df[b])

    results.append(
        {
            "Variable 1": a,
            "Variable 2": b,
            "Cramér's V": c_v,
            "% coincidencia": match_pct,
            "Tabla cruzada": ctab,
        }
    )

# Resumen numérico
summary_df = pd.DataFrame(
    [
        {
            "Variable 1": r["Variable 1"],
            "Variable 2": r["Variable 2"],
            "Cramér's V": r["Cramér's V"],
            "% coincidencia": r["% coincidencia"],
        }
        for r in results
    ]
)
# --- End of code to define summary_df ---

# =========================
# 6.1. Visualización de Cramér's V
# =========================

plt.figure(figsize=(10, 6))
sns.barplot(x="Cramér's V", y="Variable 1", hue="Variable 2", data=summary_df.sort_values("Cramér's V", ascending=False))
plt.title("Asociación entre variables de texto (Cramér's V)")
plt.xlabel("Cramér's V")
plt.ylabel("Variables")
plt.legend(title="Par de variables")
plt.grid(axis='x', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

# =========================
# 6.2. Visualización de % coincidencia
# =========================

plt.figure(figsize=(10, 6))
sns.barplot(x="% coincidencia", y="Variable 1", hue="Variable 2", data=summary_df.sort_values("% coincidencia", ascending=False))
plt.title("Porcentaje de Coincidencia entre variables de texto")
plt.xlabel("Porcentaje de Coincidencia")
plt.ylabel("Variables")
plt.legend(title="Par de variables")
plt.grid(axis='x', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

#Conclusion final

El dataset original es adecuado para analizar patrones de reseñas negativas y construir indicadores de riesgo asociados al desempeño de productos. Contiene texto, ratings, fechas, productos y categorías, lo cual permite generar variables útiles para evaluar la percepción del consumidor.
Sin embargo, el dataset tiene una limitación fundamental: no incluye datos reales de ventas. Por lo tanto, el proyecto no podrá predecir una caída de ventas en sentido estricto, sino un riesgo de caída de ventas basado en señales indirectas de insatisfacción del cliente.
Los principales problemas identificados son el desbalance de clases, la alta concentración de reseñas positivas, la presencia de variables con muchos nulos, la alta cardinalidad en variables de producto y usuario, variables potencialmente redundantes y escalas inconsistentes entre variables numéricas.
A pesar de estas limitaciones, el dataset es utilizable si se redefine correctamente el problema como una predicción de riesgo. Para ello, será necesario construir variables agregadas por producto y periodo, identificar reseñas negativas, analizar cambios en el rating promedio y usar modelos capaces de detectar patrones asociados a deterioro en la percepción del producto.


#**Ingeniería de características estándar**

A continuación presentaré 3 propuestas en Python, dependientes entre sí para crear features estándar usando su dataset de reseñas de Amazon, adaptado al nuevo enfoque: predecir riesgo de caída de ventas de un producto con base en reseñas negativas.


Limpieza inicial

In [ ]:
import pandas as pd
import numpy as np

# =========================
# 1. Cargar dataset original
# =========================

df = pd.read_csv("Amazon_Consumer_Review.csv")

print("DATASET ORIGINAL")
print("Shape:", df.shape)
display(df.head())

# =========================
# 2. Limpieza inicial
# =========================

# Convertir fechas
df["reviews.date"] = pd.to_datetime(df["reviews.date"], errors="coerce")

# Convertir numHelpful a numérica
df["reviews.numHelpful"] = pd.to_numeric(
    df["reviews.numHelpful"],
    errors="coerce"
).fillna(0)

# Eliminar columnas con demasiados nulos, si existen
columnas_eliminar = ["reviews.didPurchase", "reviews.id"]

df = df.drop(
    columns=[col for col in columnas_eliminar if col in df.columns],
    errors="ignore"
)

# =========================
# 3. Features básicas por reseña
# =========================

# Texto combinado
df["review_full_text"] = (
    df["reviews.title"].astype(str) + " " + df["reviews.text"].astype(str)
)

# Longitud del texto
df["review_length_chars"] = df["review_full_text"].apply(len)
df["review_length_words"] = df["review_full_text"].apply(lambda x: len(x.split()))

# Variables temporales
df["review_year"] = df["reviews.date"].dt.year
df["review_month"] = df["reviews.date"].dt.month
df["review_quarter"] = df["reviews.date"].dt.quarter
df["year_month"] = df["reviews.date"].dt.to_period("M").astype(str)

# Variables binarias de satisfacción
df["is_negative_review"] = np.where(df["reviews.rating"] <= 2, 1, 0)
df["is_neutral_review"] = np.where(df["reviews.rating"] == 3, 1, 0)
df["is_positive_review"] = np.where(df["reviews.rating"] >= 4, 1, 0)

# Variable de no recomendación
df["not_recommended"] = np.where(df["reviews.doRecommend"] == False, 1, 0)

print("DESPUÉS DE FEATURES POR RESEÑA")
print("Shape:", df.shape)
display(df[[
    "name",
    "reviews.rating",
    "reviews.date",
    "review_length_words",
    "is_negative_review",
    "not_recommended",
    "year_month"
]].head())

##Propuesta 1. Features agregadas por producto

In [ ]:
product_features = df.groupby("name").agg(
    total_reviews=("reviews.rating", "count"),
    avg_rating=("reviews.rating", "mean"),
    min_rating=("reviews.rating", "min"),
    max_rating=("reviews.rating", "max"),
    negative_reviews=("is_negative_review", "sum"),
    neutral_reviews=("is_neutral_review", "sum"),
    positive_reviews=("is_positive_review", "sum"),
    not_recommended_count=("not_recommended", "sum"),
    avg_helpful_votes=("reviews.numHelpful", "mean"),
    total_helpful_votes=("reviews.numHelpful", "sum"),
    avg_review_length_words=("review_length_words", "mean"),
    first_review_date=("reviews.date", "min"),
    last_review_date=("reviews.date", "max")
).reset_index()

# =========================
# Añadir Brand y Primary Categories a product_features
# =========================
# Estas columnas son atributos a nivel de producto.
product_brand_category = df[['name', 'brand', 'primaryCategories']].drop_duplicates(subset=['name'])

# Fusionar con product_features
product_features = product_features.merge(
    product_brand_category,
    on='name',
    how='left'
)

# Ratios por producto
product_features["negative_review_rate"] = (
    product_features["negative_reviews"] / product_features["total_reviews"]
)

product_features["positive_review_rate"] = (
    product_features["positive_reviews"] / product_features["total_reviews"]
)

product_features["not_recommended_rate"] = (
    product_features["not_recommended_count"] / product_features["total_reviews"]
)

# Antigüedad en días
product_features["review_period_days"] = (
    product_features["last_review_date"] - product_features["first_review_date"]
).dt.days

# Frecuencia de reseñas
product_features["review_frequency"] = (
    product_features["total_reviews"] /
    product_features["review_period_days"].replace(0, np.nan)
)

print("FEATURES POR PRODUCTO")
print("Shape:", product_features.shape)
display(product_features.head())

##Propuesta 2: Features temporales producto-mes

In [ ]:
monthly_product = df.groupby(["name", "year_month"]).agg(
    monthly_reviews=("reviews.rating", "count"),
    monthly_avg_rating=("reviews.rating", "mean"),
    monthly_negative_reviews=("is_negative_review", "sum"),
    monthly_positive_reviews=("is_positive_review", "sum"),
    monthly_not_recommended=("not_recommended", "sum"),
    monthly_avg_helpful=("reviews.numHelpful", "mean"),
    monthly_avg_length_words=("review_length_words", "mean")
).reset_index()

# =========================
# 3. Añadir Brand y Primary Categories
# =========================
# Estas variables son a nivel de producto y se pueden añadir directamente.
monthly_product = monthly_product.merge(
    product_features[['name', 'brand', 'primaryCategories']],
    on='name',
    how='left'
)

# Ratios mensuales
monthly_product["monthly_negative_rate"] = (
    monthly_product["monthly_negative_reviews"] / monthly_product["monthly_reviews"]
)

monthly_product["monthly_positive_rate"] = (
    monthly_product["monthly_positive_reviews"] / monthly_product["monthly_reviews"]
)

monthly_product["monthly_not_recommended_rate"] = (
    monthly_product["monthly_not_recommended"] / monthly_product["monthly_reviews"]
)

# Orden temporal
monthly_product["year_month_date"] = pd.to_datetime(monthly_product["year_month"])
monthly_product = monthly_product.sort_values(["name", "year_month_date"])

# Lags
monthly_product["prev_month_avg_rating"] = (
    monthly_product.groupby("name")["monthly_avg_rating"].shift(1)
)

monthly_product["prev_month_negative_rate"] = (
    monthly_product.groupby("name")["monthly_negative_rate"].shift(1)
)

# Diferencias
monthly_product["rating_change"] = (
    monthly_product["monthly_avg_rating"] - monthly_product["prev_month_avg_rating"]
)

monthly_product["negative_rate_change"] = (
    monthly_product["monthly_negative_rate"] - monthly_product["prev_month_negative_rate"]
)

# Rolling windows de 3 meses
monthly_product["rolling_3m_avg_rating"] = (
    monthly_product.groupby("name")["monthly_avg_rating"]
    .transform(lambda x: x.rolling(window=3, min_periods=1).mean())
)

monthly_product["rolling_3m_negative_rate"] = (
    monthly_product.groupby("name")["monthly_negative_rate"]
    .transform(lambda x: x.rolling(window=3, min_periods=1).mean())
)

print("FEATURES PRODUCTO-MES")
print("Shape:", monthly_product.shape)
display(monthly_product.head(10))

##Propuesta 3: Combinar features de producto con producto-mes

In [ ]:
final_dataset = monthly_product.merge(
    product_features,
    on="name",
    how="left"
)

print("DATASET FINAL COMBINADO")
print("Shape:", final_dataset.shape)
print("Columnas:", final_dataset.columns.tolist())
display(final_dataset.head())

##Crear variable proxy de riesgo

Como el dataset no tiene ventas reales, se crea una variable proxy llamada sales_drop_risk.

In [ ]:
final_dataset["sales_drop_risk"] = np.where(
    (
        (final_dataset["rating_change"] <= -0.5) |
        (final_dataset["negative_rate_change"] >= 0.10) |
        (final_dataset["rolling_3m_negative_rate"] >= 0.15) |
        (final_dataset["monthly_avg_rating"] < 4.0)
    ),
    1,
    0
)

print("DISTRIBUCIÓN DEL TARGET")
display(final_dataset["sales_drop_risk"].value_counts())
display((final_dataset["sales_drop_risk"].value_counts(normalize=True) * 100).round(2))

display(final_dataset[[
    "name",
    "year_month",
    "monthly_reviews",
    "monthly_avg_rating",
    "monthly_negative_rate",
    "rating_change",
    "negative_rate_change",
    "rolling_3m_negative_rate",
    "sales_drop_risk"
]].head(15))

In [ ]:
final_dataset.head()

#**Procesamiento y transformación de variables**

##Tratamiento híbrido de nulos en final_dataset

Estrategia aplicada: mediana en numéricas, moda en categóricas y creación de indicadores de nulos.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer

# =========================
# Tratamiento híbrido de nulos
# =========================

nulls_before = final_dataset.isnull().sum()
nulls_before_pct = (final_dataset.isnull().mean() * 100).round(2)
null_report_before = pd.DataFrame({
    "nulos_antes": nulls_before,
    "%_nulos_antes": nulls_before_pct
}).sort_values("nulos_antes", ascending=False)

print("RESUMEN DE NULOS ANTES DE IMPUTAR")
display(null_report_before[null_report_before["nulos_antes"] > 0])

# 1) Crear variables indicadoras de nulos (solo columnas con al menos un nulo)
cols_to_create_indicators = final_dataset.columns[final_dataset.isnull().any()].tolist()
for col in cols_to_create_indicators:
    final_dataset[f"{col}_is_null"] = final_dataset[col].isnull().astype(int)

# 2) Imputar numéricas con mediana y categóricas con moda utilizando un Pipeline

# Identify numeric and categorical columns for imputation (excluding the newly created indicator columns)
# We look at the original data types to ensure correct imputer strategy

# First, identify all numeric and categorical columns *after* indicator creation.
# The indicator columns are numeric and should not contain nulls, so SimpleImputer won't affect them.
all_numeric_cols = final_dataset.select_dtypes(include=np.number).columns.tolist()
all_categorical_cols = final_dataset.select_dtypes(exclude=np.number).columns.tolist()

# Filter to only include columns that actually need imputation
imputation_num_cols = [col for col in all_numeric_cols if final_dataset[col].isnull().any()]
imputation_cat_cols = [col for col in all_categorical_cols if final_dataset[col].isnull().any()]

# Create preprocessors for numerical and categorical features
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median'))
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent'))
])

# Create a ColumnTransformer to apply different transformations to different columns
# 'remainder="passthrough"' ensures that columns not explicitly listed (e.g., already filled, datetime, or target) are kept
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, imputation_num_cols),
        ('cat', categorical_transformer, imputation_cat_cols)
    ],
    remainder='passthrough'
)

# Create the full imputation pipeline
imputation_pipeline = Pipeline(steps=[('preprocessor', preprocessor)])

# Fit and transform the data using the pipeline
transformed_data = imputation_pipeline.fit_transform(final_dataset)

# Reconstruct the DataFrame with original column names
# The order of columns in transformed_data is: imputed_num_cols, imputed_cat_cols, then passthrough_cols
# Need to get the correct order of all columns after ColumnTransformer

# Get names of columns that were passed through
passthrough_cols = [col for col in final_dataset.columns if col not in (imputation_num_cols + imputation_cat_cols)]

# Reconstruct the DataFrame with the correct column order and names
final_dataset_reconstructed = pd.DataFrame(
    transformed_data,
    columns=imputation_num_cols + imputation_cat_cols + passthrough_cols,
    index=final_dataset.index
)

# Assign the reconstructed DataFrame back to final_dataset
final_dataset = final_dataset_reconstructed

# Ensure data types are consistent, especially for imputed numerical columns that might have become object
for col in imputation_num_cols:
    if col in final_dataset.columns:
        final_dataset[col] = pd.to_numeric(final_dataset[col], errors='coerce')

# Final check and report
nulls_after = final_dataset.isnull().sum()
nulls_after_pct = (final_dataset.isnull().mean() * 100).round(2)
null_report_after = pd.DataFrame({
    "nulos_despues": nulls_after,
    "%_nulos_despues": nulls_after_pct
})

null_comparison = pd.concat([
    null_report_before,
    null_report_after
], axis=1).fillna(0)
print("COMPARACIÓN DE NULOS: ANTES vs DESPUÉS")
display(null_comparison.sort_values("nulos_antes", ascending=False).head(20))

print("VALIDACIÓN FINAL")
print("Total de nulos antes:", int(nulls_before.sum()))
print("Total de nulos después:", int(nulls_after.sum()))
print("Indicadores creados:", len(cols_to_create_indicators))


In [ ]:
final_dataset.head()

### Eliminación de columnas duplicadas

Se eliminan las columnas con sufijo `_y` (`brand_y` y `primaryCategories_y`) que son duplicados de `brand_x` y `primaryCategories_x` provenientes de la tabla de `product_features`.

In [ ]:
# Identificar y eliminar columnas con sufijo '_y'
cols_to_drop_y = [col for col in final_dataset.columns if col.endswith('_y')]

if cols_to_drop_y:
    final_dataset = final_dataset.drop(columns=cols_to_drop_y)
    print(f"Columnas eliminadas: {cols_to_drop_y}")
else:
    print("No se encontraron columnas con sufijo '_y' para eliminar.")

print("Columnas después de eliminar duplicados:")
print(final_dataset.columns.tolist())
display(final_dataset.head())

Ahora, el `final_dataset` ya no contiene las columnas duplicadas. Proseguiremos con la codificación de las variables categóricas, asegurando que `brand_x` y `primaryCategories_x` sean incluidas.

#**Escalado y normalización**

Utilizando •	RobustScaler

In [ ]:
from sklearn.preprocessing import RobustScaler
import numpy as np
import pandas as pd

# =========================
# 1. Definir target
# =========================

target_col = "sales_drop_risk"

# =========================
# 2. Seleccionar columnas numéricas excluyendo el target
# =========================

numeric_cols = final_dataset.select_dtypes(include=[np.number]).columns.tolist()

# Excluir target si está dentro de las columnas numéricas
numeric_cols = [col for col in numeric_cols if col != target_col]

print(f"Columnas numéricas a escalar: {len(numeric_cols)}")
print("Target excluido:", target_col)
print("Columnas escaladas:")
print(numeric_cols)

# =========================
# 3. Crear copia del dataset
# =========================

final_dataset_robust = final_dataset.copy()

# =========================
# 4. Aplicar RobustScaler
# =========================

robust_scaler = RobustScaler()

final_dataset_robust[numeric_cols] = robust_scaler.fit_transform(
    final_dataset[numeric_cols]
)

# =========================
# 5. Evidencia antes y después
# =========================

comparacion_robust = pd.DataFrame({
    "original_median": final_dataset[numeric_cols].median(),
    "robust_median": final_dataset_robust[numeric_cols].median(),
    "original_iqr": final_dataset[numeric_cols].quantile(0.75) - final_dataset[numeric_cols].quantile(0.25),
    "robust_iqr": final_dataset_robust[numeric_cols].quantile(0.75) - final_dataset_robust[numeric_cols].quantile(0.25),
    "original_min": final_dataset[numeric_cols].min(),
    "robust_min": final_dataset_robust[numeric_cols].min(),
    "original_max": final_dataset[numeric_cols].max(),
    "robust_max": final_dataset_robust[numeric_cols].max()
})

display(comparacion_robust.head(15))

# =========================
# 6. Verificar que el target no fue escalado
# =========================

if target_col in final_dataset.columns:
    print("Valores únicos del target original:")
    print(final_dataset[target_col].unique())

    print("Valores únicos del target en dataset escalado:")
    print(final_dataset_robust[target_col].unique())

# =========================
# 7. Vista del dataset escalado
# =========================

print("Shape original:", final_dataset.shape)
print("Shape escalado:", final_dataset_robust.shape)

display(final_dataset_robust.head())

#**Codificación de variables categóricas**
Aplicando **One-Hot Encoding** sobre las variables categóricas de `final_dataset`.


In [ ]:
from sklearn.preprocessing import OneHotEncoder

# =========================
# 1. Identificar columnas categóricas
# =========================

categorical_cols = final_dataset.select_dtypes(include=["object", "category"]).columns.tolist()

print(f"Columnas categóricas encontradas: {len(categorical_cols)}")
print(categorical_cols)

# =========================
# 2. Aplicar One-Hot Encoding
# =========================

encoder = OneHotEncoder(sparse_output=False, handle_unknown="ignore")
encoded_array = encoder.fit_transform(final_dataset[categorical_cols])
encoded_feature_names = encoder.get_feature_names_out(categorical_cols)

encoded_df = pd.DataFrame(
    encoded_array,
    columns=encoded_feature_names,
    index=final_dataset.index
)

# =========================
# 3. Combinar con variables numéricas
# =========================

final_dataset_ohe = pd.concat(
    [final_dataset.drop(columns=categorical_cols), encoded_df],
    axis=1
)

print("Shape original:", final_dataset.shape)
print("Shape con One-Hot Encoding:", final_dataset_ohe.shape)
display(final_dataset_ohe.head())

### Agregar totales anuales al `final_dataset`

Para que los totales de reseñas por año estén disponibles en cada fila del dataset, se realiza un merge con el dataframe `reviews_by_year`.

In [ ]:
final_dataset_ohe['year'] = final_dataset_ohe['year_month_date'].dt.year

reviews_by_year = final_dataset_ohe.groupby('year').agg(
    total_negative_reviews=('negative_reviews', 'sum'),
    total_positive_reviews=('positive_reviews', 'sum'),
    total_neutral_reviews=('neutral_reviews', 'sum')
).reset_index()

final_dataset_ohe = final_dataset_ohe.merge(
    reviews_by_year,
    on='year',
    how='left'
)

print("Columnas de final_dataset después de agregar los totales anuales:")
display(final_dataset[['year', 'total_negative_reviews', 'total_positive_reviews', 'total_neutral_reviews']].head())

In [ ]:
display(final_dataset_ohe.head())

In [ ]:
from google.colab import files

# Guardar el dataset generado como archivo CSV
final_dataset_ohe.to_csv("final_dataset_ohe.csv", index=False)

# Descargar el archivo a su computadora
files.download("final_dataset_ohe.csv")

In [ ]:
final_dataset_robust['year'] = final_dataset_robust['year_month_date'].dt.year

reviews_by_year = final_dataset_robust.groupby('year').agg(
    total_negative_reviews=('negative_reviews', 'sum'),
    total_positive_reviews=('positive_reviews', 'sum'),
    total_neutral_reviews=('neutral_reviews', 'sum')
).reset_index()

final_dataset_robust = final_dataset_robust.merge(
    reviews_by_year,
    on='year',
    how='left'
)

print("Columnas de final_dataset después de agregar los totales anuales:")
display(final_dataset_robust[['year', 'total_negative_reviews', 'total_positive_reviews', 'total_neutral_reviews']].head())

In [ ]:
from google.colab import files

# Guardar el dataset generado como archivo CSV
final_dataset_robust.to_csv("final_dataset_robust.csv", index=False)

# Descargar el archivo a su computadora
files.download("final_dataset_robust.csv")

## Visualización de la distribución de variables clave: Original vs. One-Hot Encoded

Vamos a visualizar la distribución de algunas variables numéricas importantes de `final_dataset` y `final_dataset_ohe` mediante histogramas. Esto nos permitirá observar cómo la codificación One-Hot ha afectado al dataset, aunque las variables numéricas originales no cambian directamente por el OHE, la comparación es útil para el flujo de trabajo.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Columns to visualize (assuming these are numerical and present in both DataFrames)
plot_cols = ['monthly_reviews', 'monthly_avg_rating', 'monthly_negative_reviews', 'sales_drop_risk']

print("Histograms for final_dataset (Original Data)")
plt.figure(figsize=(16, 10))
for i, col in enumerate(plot_cols):
    plt.subplot(2, 2, i + 1)
    sns.histplot(final_dataset[col], kde=True, bins=30)
    plt.title(f'Distribution of {col} (Original)')
    plt.xlabel(col)
    plt.ylabel('Frequency')
plt.tight_layout()
plt.show()


In [ ]:
print("Histograms for final_dataset_ohe (One-Hot Encoded Data - numerical columns)")
plt.figure(figsize=(16, 10))
for i, col in enumerate(plot_cols):
    plt.subplot(2, 2, i + 1)
    sns.histplot(final_dataset_ohe[col], kde=True, bins=30)
    plt.title(f'Distribution of {col} (After OHE)')
    plt.xlabel(col)
    plt.ylabel('Frequency')
plt.tight_layout()
plt.show()


Al observar estos histogramas, notarás que las distribuciones de las variables numéricas clave (`monthly_reviews`, `monthly_avg_rating`, `monthly_negative_reviews`, `sales_drop_risk`) son idénticas en `final_dataset` y `final_dataset_ohe`. Esto es esperado, ya que la One-Hot Encoding solo afecta a las variables categóricas, creando nuevas columnas binarias sin alterar las distribuciones de las variables numéricas preexistentes.

#**Feature engineering específica del dominio**

In [ ]:
import numpy as np
import pandas as pd

# REcuerda verificar que esta cargado el dataset:
# final_dataset_ohe = pd.read_csv("final_dataset_ohe.csv")

# =========================
# 1. Evidencia antes
# =========================

print("ANTES DE CREAR LA FEATURE")
print("Shape:", final_dataset_ohe.shape)
print("Columnas existentes:")
print(final_dataset_ohe.columns.tolist())

display(
    final_dataset_ohe[
        [
            "monthly_negative_rate",
            "monthly_avg_helpful",
            "sales_drop_risk"
        ]
    ].head()
)

# =========================
# 2. Crear feature de dominio
# =========================

final_dataset_ohe["negative_helpfulness_pressure"] = (
    final_dataset_ohe["monthly_negative_rate"] *
    np.log1p(final_dataset_ohe["monthly_avg_helpful"])
)

# =========================
# 3. Crear versión binaria opcional
# =========================

# Esta variable marca los casos con presión negativa superior al percentil 75
threshold_pressure = final_dataset_ohe["negative_helpfulness_pressure"].quantile(0.75)

final_dataset_ohe["high_negative_helpfulness_pressure"] = np.where(
    final_dataset_ohe["negative_helpfulness_pressure"] > threshold_pressure,
    1,
    0
)

# =========================
# 4. Evidencia después
# =========================

print("DESPUÉS DE CREAR LA FEATURE")
print("Shape:", final_dataset_ohe.shape)

display(
    final_dataset_ohe[
        [
            "monthly_negative_rate",
            "monthly_avg_helpful",
            "negative_helpfulness_pressure",
            "high_negative_helpfulness_pressure",
            "sales_drop_risk"
        ]
    ].head(10)
)

# =========================
# 5. Estadísticas descriptivas de la nueva feature
# =========================

display(
    final_dataset_ohe[
        [
            "negative_helpfulness_pressure",
            "high_negative_helpfulness_pressure"
        ]
    ].describe()
)

# =========================
# 6. Revisar relación preliminar con el target
# =========================

display(
    final_dataset_ohe.groupby("sales_drop_risk")[
        [
            "negative_helpfulness_pressure",
            "high_negative_helpfulness_pressure"
        ]
    ].mean()
)

In [ ]:
display(final_dataset_ohe.head())

#**Selección de características**

##1. Metodo Filter

In [ ]:
from sklearn.feature_selection import VarianceThreshold

# Crear una NUEVA copia para no alterar final_dataset_ohe
final_dataset_ohe_filtered = final_dataset_ohe.copy()

# Seleccionar columnas numéricas para aplicar el filtro estadístico
numeric_cols = final_dataset_ohe_filtered.select_dtypes(include=["number"]).columns

# Variance Threshold: elimina variables con varianza muy baja (casi constantes)
selector = VarianceThreshold(threshold=0.01)
selector.fit(final_dataset_ohe_filtered[numeric_cols])

selected_numeric_cols = numeric_cols[selector.get_support()]
removed_numeric_cols = numeric_cols[~selector.get_support()]

print("Variables numéricas seleccionadas (VarianceThreshold):")
print(list(selected_numeric_cols))
print("Variables numéricas eliminadas (baja varianza):")
print(list(removed_numeric_cols))

# Mantener también columnas no numéricas para preservar contexto del dataset
non_numeric_cols = final_dataset_ohe_filtered.columns.difference(numeric_cols)
final_dataset_ohe_filtered = final_dataset_ohe_filtered.filter(
    items=list(non_numeric_cols) + list(selected_numeric_cols)
)

print("Dataset original (final_dataset_ohe):", final_dataset_ohe.shape)
print("Nuevo dataset filtrado (final_dataset_ohe_filtered):", final_dataset_ohe_filtered.shape)
print("Columnas removidas por baja varianza:", len(numeric_cols) - len(selected_numeric_cols))

# Vista previa del nuevo dataset
final_dataset_ohe_filtered.head()



##Tabla con la varianza de cada variable

In [ ]:
variance_summary = pd.DataFrame({
    "variable": numeric_cols,
    "variance": selector.variances_,
    "selected": selector.get_support()
})

variance_summary["decision"] = np.where(
    variance_summary["selected"],
    "Seleccionada",
    "Eliminada por baja varianza"
)

variance_summary = variance_summary.sort_values(by="variance")

display(variance_summary)

In [ ]:
from sklearn.feature_selection import RFECV
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold

# =========================
# Wrapper Method (RFECV) sin alterar final_dataset_ohe
# =========================

# Copia de trabajo
wrapper_base = final_dataset_ohe.copy()

# Definir target (ajustar si cambia el nombre en el dataset)
target_col = "sales_drop_risk"

# Variables candidatas: solo numéricas y excluyendo el target
X = wrapper_base.select_dtypes(include=["number"]).drop(columns=[target_col], errors="ignore")
y = wrapper_base[target_col]

# Modelo base para RFE
model = LogisticRegression(max_iter=2000, random_state=42)

# RFECV selecciona automáticamente la cantidad óptima de variables
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
selector_wrapper = RFECV(
    estimator=model,
    step=1,
    cv=cv,
    scoring="f1",
    min_features_to_select=5,
    n_jobs=-1
)

selector_wrapper.fit(X, y)

selected_features = X.columns[selector_wrapper.support_]

# NUEVO dataset resultante del método Wrapper
final_dataset_ohe_wrapper = pd.concat(
    [wrapper_base[selected_features], wrapper_base[[target_col]]],
    axis=1
)

print("Modelo utilizado:", model.__class__.__name__)
print("Métrica utilizada:", selector_wrapper.scoring)
print("Cantidad final de variables:", len(selected_features))
print("Shape del nuevo dataset (wrapper):", final_dataset_ohe_wrapper.shape)

display(final_dataset_ohe_wrapper.head())

# Gráfico: desempeño del RFECV según número de variables seleccionadas
cv_scores = selector_wrapper.cv_results_["mean_test_score"]
n_features = selector_wrapper.cv_results_["n_features"]

plt.figure(figsize=(10, 5))
plt.plot(n_features, cv_scores, marker="o", linewidth=2)
plt.axvline(selector_wrapper.n_features_, color="crimson", linestyle="--",
            label=f"Óptimo: {selector_wrapper.n_features_} variables")
plt.title("Wrapper (RFECV): F1 promedio por número de variables")
plt.xlabel("Número de variables seleccionadas")
plt.ylabel("F1 promedio (validación cruzada)")
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()


# Reducción dimensional con PCA

Aplicaremos PCA sobre `final_dataset_ohe` usando únicamente variables numéricas (excluyendo la variable objetivo `sales_drop_risk`). Luego compararemos el número de dimensiones originales vs. las dimensiones finales necesarias para conservar el 95% de la varianza y graficaremos la varianza explicada acumulada.


In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import numpy as np
import matplotlib.pyplot as plt

# Copia de trabajo para no alterar el dataset original
pca_base = final_dataset_ohe.copy()

# Seleccionar columnas numéricas y excluir target si existe
numeric_features = pca_base.select_dtypes(include=["number"]).drop(columns=["sales_drop_risk"], errors="ignore")

# Rellenar posibles nulos para evitar errores en PCA
numeric_features = numeric_features.fillna(0)

# Escalado (recomendado para PCA)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(numeric_features)

# PCA completo para analizar varianza explicada
pca_full = PCA(random_state=42)
pca_full.fit(X_scaled)

explained_variance_ratio = pca_full.explained_variance_ratio_
cumulative_explained_variance = np.cumsum(explained_variance_ratio)

# Número mínimo de componentes para conservar al menos 95% de varianza
n_components_95 = np.argmax(cumulative_explained_variance >= 0.95) + 1

# Ajustar PCA final con ese número de componentes
pca_final = PCA(n_components=n_components_95, random_state=42)
X_pca = pca_final.fit_transform(X_scaled)

original_dims = numeric_features.shape[1]
final_dims = X_pca.shape[1]

print(f"Dimensiones originales: {original_dims}")
print(f"Dimensiones finales (95% varianza): {final_dims}")
print(f"Varianza explicada acumulada con {final_dims} componentes: {pca_final.explained_variance_ratio_.sum():.4f}")

# Gráfico de varianza explicada acumulada
plt.figure(figsize=(10, 6))
plt.plot(range(1, len(cumulative_explained_variance) + 1), cumulative_explained_variance, marker='o')
plt.axhline(y=0.95, color='r', linestyle='--', label='95% varianza')
plt.axvline(x=n_components_95, color='g', linestyle='--', label=f'{n_components_95} componentes')
plt.title('PCA - Varianza explicada acumulada')
plt.xlabel('Número de componentes principales')
plt.ylabel('Varianza explicada acumulada')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()



El análisis mediante PCA permitió reducir la dimensionalidad del dataset desde 187 variables originales hasta 113 componentes principales, manteniendo aproximadamente un 95.3% de la varianza total del conjunto de datos. Esto representa una reducción cercana al 40% del espacio original de características.

La forma de la curva indica que existe redundancia y correlación entre algunas variables del dataset, especialmente debido a la presencia de variables derivadas, indicadores temporales, métricas agregadas y columnas generadas mediante One-Hot Encoding. La reducción obtenida sugiere que parte de la información original estaba distribuida entre variables relacionadas.

Además, el gráfico muestra que después de aproximadamente 113 componentes la curva comienza a estabilizarse, indicando que los componentes adicionales aportan cantidades cada vez menores de información.

Desde la perspectiva del modelado, esta reducción es beneficiosa para la Regresión Logística porque disminuye la multicolinealidad, reduce el riesgo de sobreajuste y simplifica el entrenamiento del modelo sin perder información significativa.